In [1]:
%env NODE_OPTIONS=--max-old-space-size=32768

env: NODE_OPTIONS=--max-old-space-size=32768


In [2]:
import torch
from transformers import TrainingArguments, Trainer, DataCollatorForLanguageModeling
from trl import SFTTrainer
import numpy as np
import neptune
from accelerate import Accelerator
# from accelerate import Accelerator
import evaluate
import os
# import ..utils
import sys
from datasets import Dataset
sys.path.append('/sise/home/urizlo/VuLLM_One_Stage')
from utils import Mistral_7B, Create_lora_mistral
from code_files.preprocess_data import Prepare_dataset_with_only_replace_mistral

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

In [3]:
# from transformers import AutoTokenizer, MistralForCausalLM
# device_index = Accelerator().process_index
# device_map = {"": device_index}
# model = MistralForCausalLM.from_pretrained("mistralai/Mistral-7B-v0.1", device_map=device_map, torch_dtype=torch.bfloat16)
# tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")prompt = "are you gay?"
# model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)
# generated_ids = model.generate(**model_inputs, max_new_tokens=100, do_sample=True)
# print(tokenizer.batch_decode(generated_ids)[0])

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
checkpoint = "mistralai/Mistral-7B-v0.1"
model, tokenizer = Mistral_7B.create_model_and_tokenizer(checkpoint)

Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
%cd ..
%cd ..
%cd ..

/sise/home/urizlo/VuLLM_One_Stage/code_files/fine_tuning
/sise/home/urizlo/VuLLM_One_Stage/code_files
/sise/home/urizlo/VuLLM_One_Stage


In [5]:
path_trainset = "Datasets/vulgen_train_with_diff_lines_spaces.csv"
path_testset = "Datasets/vulgen_test_with_diff_lines_spaces.csv"
full_vulgen = True

train, test = Prepare_dataset_with_only_replace_mistral.create_datasets(path_trainset, path_testset, full_vulgen)
train = train[:100]
test = test[:100]

train = Dataset.from_pandas(train)
test= Dataset.from_pandas(test)


8135
978


In [6]:
def generate_prompt(sample, return_response=True):
    prompt = f"""<s>[INST] {sample['inputs']} [/INST] \n {sample['outputs']} </s>"""
    return [prompt]

In [7]:
model = Create_lora_mistral.create_lora(model, rank=4, dropout=0.05)

trainable params: 10485760 || all params: 7252217856 || trainable%: 0.14458694165295632


In [8]:
# config evaluation metrics
metric = evaluate.load("sacrebleu")
google_bleu = evaluate.load("google_bleu")

def postprocess_text(preds, labels):
    preds = [pred.strip() for pred in preds]
    labels = [[label.strip()] for label in labels]
    return preds, labels

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)

    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    decoded_preds, decoded_labels = postprocess_text(decoded_preds, decoded_labels)
    #ScareBleu
    results = metric.compute(predictions=decoded_preds, references=decoded_labels)
    result = {"sacreBleu": results["score"]}
    #GoogleBlue
    results = google_bleu.compute(predictions=decoded_preds, references=decoded_labels)
    result["googleBleu"] = results["google_bleu"]
    #Accuracy
    count = 0
    for p, l in zip(decoded_preds, decoded_labels):
        if p == l[0]:
            count += 1
    total_tokens = len(decoded_labels)
    accuracy = count / total_tokens
    result['accuracy'] = accuracy
    #Genaration length
    prediction_lens = [np.count_nonzero(pred != tokenizer.pad_token_id) for pred in preds]
    result["gen_len"] = np.mean(prediction_lens)
    result = {k: round(v, 4) for k, v in result.items()}
    return result

In [9]:
# config env varibles
# NEPTUNE_API_TOKEN = os.environ.get("NEPTUNE_API_TOKEN")
# NEPTUNE_PROJECT = os.environ.get("NEPTUNE_PROJECT")
os.environ["NEPTUNE_API_TOKEN"] = 'eyJhcGlfYWRkcmVzcyI6Imh0dHBzOi8vYXBwLm5lcHR1bmUuYWkiLCJhcGlfdXJsIjoiaHR0cHM6Ly9hcHAubmVwdHVuZS5haSIsImFwaV9rZXkiOiI4Y2VlNTFhZC1hODJkLTQ4NzItOTE0MS0yZmNkNWY3ZWE0MTEifQ=='
os.environ["NEPTUNE_PROJECT"] = 'zlotman/Localization-model'
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "true"

model.config.num_labels = tokenizer.vocab_size

In [10]:
# create trainer object
training_args = TrainingArguments(
    output_dir="saved_models/Mistral",
    evaluation_strategy="epoch",
    learning_rate=5e-5,
    adam_beta1=0.9,
    adam_beta2=0.95,
    adam_epsilon=1e-8,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=1,
    weight_decay=0.001,
    num_train_epochs=4,
    # predict_with_generate=True,
    bf16=True,
    tf32=True,
    # remove_unused_columns=False,
    logging_dir="TensorBoard",
    do_train=True,
    do_eval=True,
    logging_strategy='epoch',
    # generation_max_length=810,
    # generation_num_beams=1,
    dataloader_num_workers=4,
    warmup_steps=57000,
    # report_to="no",
    lr_scheduler_type='linear',
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
max_seq_length = 2200
trainer = SFTTrainer(
    model=model,
    args=training_args,
    dataset_batch_size=1,
    train_dataset=train,
    data_collator=data_collator,
    eval_dataset=test,
    max_seq_length=max_seq_length,
    tokenizer=tokenizer,
    formatting_func=generate_prompt,
    compute_metrics=compute_metrics
)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

/home/urizlo/.conda/envs/vulinject/lib/python3.10/site-packages/accelerate/accelerator.py:436: FutureWarning: Passing the following arguments to `Accelerator` is deprecated and will be removed in version 1.0 of Accelerate: dict_keys(['dispatch_batches', 'split_batches', 'even_batches', 'use_seedable_sampler']). Please pass an `accelerate.DataLoaderConfiguration` instead: 
dataloader_config = DataLoaderConfiguration(dispatch_batches=None, split_batches=False, even_batches=True, use_seedable_sampler=True)
  warnings.warn(
Detected kernel version 3.10.0, which is below the recommended minimum of 5.5.0; this can cause the process to hang. It is recommended to upgrade the kernel to the minimum version or higher.


In [ ]:
for i, batch in enumerate(trainer.get_train_dataloader()):
    if i == 2:
        print(batch)
        break

In [11]:
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
# os.environ['TORCH_USE_CUDA_DSA'] = '1'
from torch.cuda.amp import autocast, GradScaler

scaler = GradScaler()

for i, batch in enumerate(trainer.get_train_dataloader()):
    # batch['input_ids'] = torch.tensor(batch['input_ids'], dtype=torch.long).unsqueeze(0).to(model.device)
    # batch['attention_mask'] = torch.tensor(batch['attention_mask'], dtype=torch.bfloat16).unsqueeze(0).to(model.device)
    # batch['labels'] = torch.tensor(batch['labels'], dtype=torch.long).unsqueeze(0).to(model.device)
        # Check labels for any out-of-bound values
    # batch['labels'] = batch['labels'].where(batch['labels'] != -100, 2)
    # batch['labels'] = torch.ones_like(batch['labels'])
    # batch['input_ids'] = torch.ones_like(batch['input_ids'])
    print(batch['labels'])
    print("Label max:", batch['labels'].max())
    print("Label min:", batch['labels'].min())

    # Assuming 'num_classes' is the number of classes your model outputs
    num_classes = model.config.num_labels  # Adjust this according to your model's specific attribute or configuration
    print(num_classes)
    # assert (batch['labels'] >= 0).all() and (batch['labels'] < num_classes).all(), "Labels are out of bound."
    with autocast():
        outputs = model(**batch)
    # scaler.scale(loss).backward()
    # model.optimizer.step()
    # scaler.update()
    # model.optimizer.zero_grad()
    if i > 1:
        break  # just a quick test of a few batches

tensor([[    1,     1,   733, 16289, 28793,  5936,  5392, 10041,  5546,   398,
           769, 14868, 28730,  6465,   470,  2532, 28711,  2287,   513,  1661,
           769, 14868, 28730,  6465,  2769,   527,  1904,  2532, 28711,  5390,
          4400, 28730,  6465,  2769,  7159, 28732,  1095, 17697,  5546, 28732,
          8903, 28730,  1724,  1648,  7741,  1648,  7683, 28730, 15590, 28730,
          3789, 28730,  6465,  1648,  4686, 28730, 15931,  1648,  6855,  1648,
           268,  2181,  1648,  5813, 28730,  5987,  8797, 16076, 28711,  2287,
           604,  4400, 28730,  6465,  2769,   527,   692,   443,  1421,   733,
         28748, 16289, 28793, 28705,    13,  5936,     1,   414, 28711,   273,
          4400, 28730,  6465,  2769,  7159, 28732,  1095, 17697,  5546, 28732,
          8903, 28730,  1724,  1648,  7741,  1648,  7683, 28730, 15590, 28730,
          3789, 28730,  6465,  1648,  4686, 28730, 15931,  1648,  6855,  1648,
           268,  2181,  1648,  5813, 28730,  5987,  

OutOfMemoryError: CUDA out of memory. Tried to allocate 112.00 MiB. GPU 0 has a total capacty of 47.51 GiB of which 79.44 MiB is free. Including non-PyTorch memory, this process has 47.43 GiB memory in use. Of the allocated memory 45.23 GiB is allocated by PyTorch, and 798.43 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [11]:
trainer.train()

/home/urizlo/.conda/envs/vulinject/lib/python3.10/site-packages/neptune/common/warnings.py:62: NeptuneWarning: The following monitoring options are disabled by default in interactive sessions: 'capture_stdout', 'capture_stderr', 'capture_traceback', and 'capture_hardware_metrics'. To enable them, set each parameter to 'True' when initializing the run. The monitoring will continue until you call run.stop() or the kernel stops. Also note: Your source files can only be tracked if you pass the path(s) to the 'source_code' argument. For help, see the Neptune docs: https://docs.neptune.ai/logging/source_code/
  warnings.warn(


https://app.neptune.ai/zlotman/Localization-model/e/LOC-255
Experiencing connection interruptions. Will try to reestablish communication with Neptune. Internal exception was: HTTPTooManyRequests
Communication with Neptune restored!


Exception ignored in: <bound method MetadataContainer._before_fork of <neptune.metadata_containers.run.Run object at 0x7fdc3de9df30>>
Traceback (most recent call last):
  File "/home/urizlo/.conda/envs/vulinject/lib/python3.10/site-packages/neptune/metadata_containers/metadata_container.py", line 265, in _before_fork
    self._op_processor.pause()
  File "/home/urizlo/.conda/envs/vulinject/lib/python3.10/site-packages/neptune/internal/operation_processors/async_operation_processor.py", line 159, in pause
    self._consumer.pause()
  File "/home/urizlo/.conda/envs/vulinject/lib/python3.10/site-packages/neptune/internal/threading/daemon.py", line 55, in pause
    self._wait_condition.wait_for(lambda: self._state != Daemon.DaemonState.PAUSING)
  File "/home/urizlo/.conda/envs/vulinject/lib/python3.10/threading.py", line 355, in wait_for
    self.wait(waittime)
  File "/home/urizlo/.conda/envs/vulinject/lib/python3.10/threading.py", line 320, in wait
    waiter.acquire()
KeyboardInterrupt:

Experiencing connection interruptions. Will try to reestablish communication with Neptune. Internal exception was: HTTPTooManyRequests
